# FastAPI Runner — Colab T4 + ngrok

Lokal 4 GB VRAM kısıtı için Qwen2.5-VL-7B'yi Colab T4'te çalıştıran köprü.

```
[Lokal PyCharm] → git push → [GitHub] → git pull → [Colab T4]
                                                       │
                                                       ├── uvicorn (FastAPI + Qwen + SBERT)
                                                       └── ngrok tunnel → public URL
                                                                          │
                                                                          ↓
                                                                  [Lokal Spring Boot]
```

**Çalıştırmadan önce:**
1. Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**
2. SBERT modeli Drive'da olmalı: `MyDrive/tez_kaynaklari/model_ai_noted_with_negatives_positives_v2/`
3. ngrok bedava token (https://dashboard.ngrok.com/get-started/your-authtoken)

**En basit kullanım:** Çalışma zamanı → **Tümünü çalıştır**. Drive izni, ngrok token, görsel yükleme için tek tek sorulur.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader

## 2. Repo'yu çek

In [ ]:
import os

REPO_URL = 'https://github.com/nebiavsar/tez_fastAPI.git'
REPO_DIR = '/content/tez_fastAPI'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 3. Bağımlılıklar

`pillow<12` kritik — Colab pre-installed torchvision Pillow 12.x ile uyumsuz.

In [ ]:
!pip install -q -r requirements.txt pyngrok nest-asyncio

In [ ]:
import torch, transformers, bitsandbytes
print(f'torch        : {torch.__version__} (cuda={torch.cuda.is_available()})')
print(f'transformers : {transformers.__version__}')
print(f'bitsandbytes : {bitsandbytes.__version__}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB')

## 4. Drive bağla + SBERT modelini kopyala

SBERT modeli 449 MB → GitHub'da yok (.gitignore). Drive'dan çekilir.

**İlk kullanımda:** `https://drive.google.com` aç → `tez_kaynaklari` klasörü yarat → lokal'deki `model_ai_noted_with_negatives_positives_v2/` klasörünü içine sürükle.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_SBERT = '/content/drive/MyDrive/tez_kaynaklari/model_ai_noted_with_negatives_positives_v2'
LOCAL_SBERT = '/content/tez_fastAPI/model_ai_noted_with_negatives_positives_v2'

if not os.path.exists(DRIVE_SBERT):
    raise FileNotFoundError(f'SBERT modeli Drive\'da yok: {DRIVE_SBERT}')

if not os.path.exists(LOCAL_SBERT):
    print('SBERT kopyalanıyor...')
    shutil.copytree(DRIVE_SBERT, LOCAL_SBERT)
    print('Tamam.')
else:
    print('SBERT zaten var, atlanıyor.')

## 4b. Qwen cache restore (Drive → local)

**Akıllı tasarım:** Eğer Qwen daha önce Drive'a yedeklendiyse, oradan kopyalanır → 6 GB indirme YOK. Yedek yoksa Bölüm 5'te indirilecek.

**Tek seferlik bedel, kalıcı çözüm:** İlk indirmeden sonra Bölüm 5b'de Drive'a yedek alıyoruz.

In [ ]:
DRIVE_QWEN = '/content/drive/MyDrive/tez_kaynaklari/qwen_cache'
LOCAL_QWEN = '/root/.cache/huggingface/hub'

if os.path.exists(LOCAL_QWEN) and any(os.scandir(LOCAL_QWEN)):
    print('Qwen cache zaten /root\'ta — atlanıyor.')
elif os.path.exists(DRIVE_QWEN):
    print('Qwen cache Drive\'dan kopyalanıyor (~3-5 dk)...')
    os.makedirs(os.path.dirname(LOCAL_QWEN), exist_ok=True)
    shutil.copytree(DRIVE_QWEN, LOCAL_QWEN, dirs_exist_ok=True)
    print('Qwen cache restore tamam. Bölüm 5\'te indirme YOK.')
else:
    print('Qwen Drive\'da yok — Bölüm 5\'te ilk indirme yapılacak (~6 GB, ~10 dk).')
    print('İndirmeden sonra Bölüm 5b ile Drive\'a yedekle.')

## 5. Uvicorn'u arka planda başlat

FastAPI lifespan event'i Qwen + SBERT yükler.

**Cache'liyse:** ~60-90 sn (RAM'e yükle)
**Cache yoksa:** ~10 dk (6 GB indirme + yükleme)

In [ ]:
import subprocess, time
from pathlib import Path

LOG_FILE = Path('/content/uvicorn.log')
LOG_FILE.unlink(missing_ok=True)

uvicorn_proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd='/content/tez_fastAPI',
    stdout=open(LOG_FILE, 'w'),
    stderr=subprocess.STDOUT,
)
print(f'Uvicorn PID: {uvicorn_proc.pid}')
print('Bekleniyor — "sunucu hazır" mesajı görünene kadar...')

deadline = time.time() + 1200  # 20 dk max
while time.time() < deadline:
    if LOG_FILE.exists():
        content = LOG_FILE.read_text(errors='ignore')
        if 'sunucu hazır' in content:
            print('✓ Uvicorn hazır.')
            break
        if 'Traceback' in content:
            print('HATA:')
            print(content[-2000:])
            break
    time.sleep(15)
    elapsed = int(time.time() - (deadline - 1200))
    print(f'  ..bekleniyor ({elapsed}s)')

## 5b. Qwen cache backup (local → Drive)

**Tek seferlik:** Qwen ilk kez inip yüklendikten sonra cache'i Drive'a kopyala. Sonraki runtime reset'lerinde Bölüm 4b oradan restore eder → indirme YOK.

Drive'da zaten varsa atlanır.

In [ ]:
if os.path.exists(DRIVE_QWEN):
    print('Qwen cache Drive\'da zaten var — atlanıyor.')
elif os.path.exists(LOCAL_QWEN) and any(os.scandir(LOCAL_QWEN)):
    print('Qwen cache Drive\'a yedekleniyor (~3-5 dk)...')
    shutil.copytree(LOCAL_QWEN, DRIVE_QWEN)
    print('Tamam. Sonraki runtime reset\'lerinde 6 GB indirme YOK.')
else:
    print('Qwen cache local\'de yok — Bölüm 5\'i çalıştırdın mı?')

## 6. ngrok tunnel

İlk seferinde token isteyecek (https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
from pyngrok import ngrok, conf
from getpass import getpass

if not conf.get_default().auth_token:
    token = getpass('ngrok auth token: ')
    ngrok.set_auth_token(token)

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print('=' * 70)
print(f'  Public URL: {PUBLIC_URL}')
print(f'  Swagger:    {PUBLIC_URL}/docs')
print('=' * 70)
print()
print('Spring Boot için (IntelliJ Run Config > Environment variables):')
print(f'  FASTAPI_BASE_URL={PUBLIC_URL}')
print()
print('VEYA Windows PowerShell\'de Spring Boot başlatmadan önce:')
print(f'  $env:FASTAPI_BASE_URL="{PUBLIC_URL}"')
print(f'  mvn spring-boot:run    # veya IntelliJ\'den Run')
print()
print('Bu URL ngrok bedava tier — Colab session her yenilendiğinde değişir.')

## 7. Smoke test — /health

In [ ]:
import requests
r = requests.get(f'{PUBLIC_URL}/health', timeout=10)
print(f'Status: {r.status_code}')
print(f'Body:   {r.json()}')

## 8. End-to-end test — /process-exam

**İki görsel** gerekir:
- `paperImage`: öğrenci kâğıdı
- `answerKeyImage`: öğretmen cevap kâğıdı

Pipeline testi için `ornek3.jpeg`'i iki kez yükleyebilirsin.

In [ ]:
from google.colab import files

print('═══ 1) ÖĞRENCİ kâğıdını yükle ═══')
uploaded_paper = files.upload()
paper_filename = list(uploaded_paper.keys())[0]
print(f'  → {paper_filename}\n')

print('═══ 2) ÖĞRETMEN CEVAP kâğıdını yükle ═══')
uploaded_ak = files.upload()
answer_key_filename = list(uploaded_ak.keys())[0]
print(f'  → {answer_key_filename}')

In [ ]:
import json

print('Çağrı gönderiliyor (~3-5 dk)...')
with open(paper_filename, 'rb') as pf, open(answer_key_filename, 'rb') as af:
    r = requests.post(
        f'{PUBLIC_URL}/process-exam',
        files={
            'paperImage': (paper_filename, pf, 'image/jpeg'),
            'answerKeyImage': (answer_key_filename, af, 'image/jpeg'),
        },
        timeout=600,
    )

print(f'\nStatus: {r.status_code}')
if r.status_code != 200:
    print(r.text[:2000])
else:
    result = r.json()
    print(f'\nToplam skor: {result["score"]}\n')
    for q in result['questions']:
        print(f'═══ Soru {q["questionId"]} (skor: {q["score"]}) ═══')
        print(f'  Öğrenci  : {q["extractedAnswer"]}')
        print(f'  Referans : {q["expectedAnswer"]}')
        print(f'  Geri bld : {q["feedback"]}\n')

## 9. Tip tespitini log'dan doğrula

Heuristic'in (Python pattern matching) her sorunun tipini doğru atayıp atamadığını gör.

In [ ]:
log = open('/content/uvicorn.log').read()
for line in log.split('\n'):
    if 'tipler:' in line:
        print(line)